# Experiment 05: Lag Boundary

Sweep lag relative to theory dimension and check the transition around L / (2k) ~ 1.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "config.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


In [ ]:
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable if iterable is not None else []

from src import ExperimentConfig, run_experiment


def overall_row(results: dict) -> pd.Series:
    summary_df = results["summary_df"]
    return summary_df.loc[summary_df["set_idx"] == "overall"].iloc[0]


def collect_overall_rows(result_map: dict[str, dict], columns: list[str] | None = None) -> pd.DataFrame:
    rows = []
    for run_label, result in result_map.items():
        row = overall_row(result).copy()
        row["run_label"] = run_label
        rows.append(row)
    df = pd.DataFrame(rows)
    if columns is not None:
        return df[["run_label", *columns]]
    return df


In [ ]:
def safe_min_delta_theta(base_config: dict, num_freqs: int, slack: float = 0.98, requested: float = 0.04 * np.pi) -> float:
    requested = base_config.get("MIN_DELTA_THETA", requested)
    if num_freqs <= 1:
        return requested
    available_span = base_config["theta_max"] - base_config["theta_min"]
    max_feasible = slack * available_span / (num_freqs - 1)
    return min(requested, max_feasible)


BASE_CONFIG = dict(
    time_mode="discrete",
    theta_min=0.05 * np.pi,
    theta_max=0.85 * np.pi,
    RANDOM_AMPLITUDE=False,
    RANDOM_PHASE=True,
    USE_NOISE=False,
    HIDDEN_DIM=128,
    NUM_EXPERIMENTS=20,
    SEEDS_PER_FREQ=5,
    VAL_RATIO=0.2,
    TEST_RATIO=0.2,
    NORMALIZE_H_COLUMNS=False,
    VERBOSE=False,
    MAKE_PLOTS=False,
)


In [ ]:
def run_lag_boundary(num_freqs: int, lag: int, bottleneck_multiplier: int, model_id: str, lr: float) -> dict:
    cfg = ExperimentConfig(
        NUM_FREQS=num_freqs,
        LAG=lag,
        BOTTLENECK_MULTIPLIER=bottleneck_multiplier,
        SEQ_LEN=4096,
        MIN_DELTA_THETA=safe_min_delta_theta(BASE_CONFIG, num_freqs),
        MODEL_ID=model_id,
        LR=lr,
        EPOCHS=1500,
        **BASE_CONFIG,
    )
    return run_experiment(cfg)


In [ ]:
results = {}
condition_specs = [
    (num_freqs, lag, bottleneck_multiplier, model_id, lr)
    for num_freqs in [2, 4, 8, 16]
    for lag in [num_freqs, 2 * num_freqs - 1, 2 * num_freqs, 3 * num_freqs, 4 * num_freqs, 6 * num_freqs]
    for bottleneck_multiplier in [2, 4]
    for model_id, lr in [("AN003_LINEAR", 1e-2), ("AN002_NO_BN_TANH", 1e-3)]
]
for num_freqs, lag, bottleneck_multiplier, model_id, lr in tqdm(condition_specs, desc="experiment 5 conditions"):
    seq_len = 8192 if num_freqs == 16 else 4096
    cfg = ExperimentConfig(
        **BASE_CONFIG,
        SEQ_LEN=seq_len,
        NUM_FREQS=num_freqs,
        MIN_DELTA_THETA=safe_min_delta_theta(BASE_CONFIG, num_freqs),
        LAG=lag,
        BOTTLENECK_MULTIPLIER=bottleneck_multiplier,
        MODEL_ID=model_id,
        LR=lr,
        EPOCHS=1500,
    )
    label = f"model={model_id},k={num_freqs},L={lag},bmul={bottleneck_multiplier}"
    results[label] = run_experiment(cfg)

collect_overall_rows(
    results,
    columns=[
        "test_mse_mean",
        "test_align_coverage_full_mean",
        "test_recon_r2_qf_from_h_mean",
        "test_align_mean_angle_deg_full_mean",
        "train_rank_threshold_mean",
        "test_rank_threshold_mean",
        "train_rank_entropy_mean",
        "test_rank_entropy_mean",
        "train_h_numerical_dim_mean",
        "test_h_numerical_dim_mean",
    ],
)
